# **05 Final Load Prep**

Use this notebook to compute final KPIs, prepare the Tableau-ready dataset, and export the exact file used in the dashboard.

In [29]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

In [30]:

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

## **Data Loading**

In [31]:
DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'
df = pd.read_csv(DATA_PATH)

In [32]:
df.shape

(293468, 25)

In [33]:
df.head()

,Transaction_ID,Customer_ID,City,State,Country,Age,Gender,Income,Customer_Segment,Date,...,Total_Amount,Product_Category,Product_Brand,Product_Type,Feedback,Shipping_Method,Payment_Method,Order_Status,Ratings,products
0,8691788,37249,Dortmund,Berlin,Germany,21,Male,Low,Regular,2023-09-18,...,324.09,Clothing,Nike,Shorts,Excellent,Same-Day,Debit Card,Shipped,5,Cycling shorts
1,2174773,69749,Nottingham,England,UK,19,Female,Low,Premium,2023-12-31,...,806.70,Electronics,Samsung,Tablet,Excellent,Standard,Credit Card,Processing,4,Lenovo Tab
2,6679610,30192,Geelong,New South Wales,Australia,48,Male,Low,Regular,2023-04-26,...,1063.44,Books,Penguin Books,Children's,Average,Same-Day,Credit Card,Processing,2,Sports equipment
3,7232460,62101,Edmonton,Ontario,Canada,56,Male,High,Premium,2023-05-08,...,2466.87,Home Decor,Home Depot,Tools,Excellent,Standard,PayPal,Processing,4,Utility knife
4,4983775,27901,Bristol,England,UK,22,Male,Low,Premium,2024-01-10,...,248.56,Grocery,Nestle,Chocolate,Bad,Standard,Cash,Shipped,1,Chocolate cookies


## **Section 1 — Date & Time Feature Engineering**

In [34]:
# Convert Date column from string to datetime
df['Date'] = pd.to_datetime(df['Date'])

In [35]:
# Extract Month_Num so Tableau can sort months correctly (January=1, December=12)
df['Month_Num'] = df['Date'].dt.month

In [36]:
# Extract Quarter (Q1, Q2, Q3, Q4)
df['Quarter'] = df['Date'].dt.quarter.map({1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'})

In [37]:
# Extract Day of Week (Monday, Tuesday, ... Sunday)
df['Day_of_Week'] = df['Date'].dt.day_name()

In [38]:
# Extract Hour from the Time column for peak hour analysis
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour

In [39]:
# Verify new datetime columns
df[['Date', 'Month', 'Month_Num', 'Quarter', 'Day_of_Week', 'Time', 'Hour']].head(10)

,Date,Month,Month_Num,Quarter,Day_of_Week,Time,Hour
0,2023-09-18,September,9,Q3,Monday,22:03:55,22
1,2023-12-31,December,12,Q4,Sunday,08:42:04,8
2,2023-04-26,April,4,Q2,Wednesday,04:06:29,4
3,2023-05-08,May,5,Q2,Monday,14:55:17,14
4,2024-01-10,January,1,Q1,Wednesday,16:54:07,16
5,2023-09-21,September,9,Q3,Thursday,23:24:27,23
6,2023-06-26,June,6,Q2,Monday,13:35:51,13
7,2023-03-24,March,3,Q1,Friday,10:12:56,10
8,2024-01-06,January,1,Q1,Saturday,14:38:26,14
9,2023-10-04,October,10,Q4,Wednesday,22:27:40,22


### Observation
Four new time-based columns have been added to support Tableau filters and sorting.
`Month_Num` ensures months sort numerically (1–12) instead of alphabetically.
`Quarter` and `Day_of_Week` enable time-pattern analysis in the dashboard.
`Hour` enables peak transaction hour analysis.

## **Section 2 — KPI Calculations (Row-Level)**

In [40]:
# KPI 1: Revenue per Purchase — average basket value per item in a transaction
df['Revenue_per_Purchase'] = (df['Total_Amount'] / df['Total_Purchases']).round(2)

In [41]:
# KPI 2: High Value Flag — 1 if Total_Amount is above the 75th percentile, else 0
threshold_75 = df['Total_Amount'].quantile(0.75)
df['High_Value_Flag'] = (df['Total_Amount'] > threshold_75).astype(int)
print(f'75th percentile threshold: {threshold_75:.2f}')
print(f'High-value transactions: {df["High_Value_Flag"].sum():,} ({df["High_Value_Flag"].mean()*100:.1f}%)')

75th percentile threshold: 2031.33
High-value transactions: 73,366 (25.0%)


In [42]:
# KPI 3: Satisfied Flag — 1 if Ratings >= 4, else 0
df['Satisfied_Flag'] = (df['Ratings'] >= 4).astype(int)
print(f'Satisfied customers: {df["Satisfied_Flag"].sum():,} ({df["Satisfied_Flag"].mean()*100:.1f}%)')

Satisfied customers: 144,137 (49.1%)


In [43]:
# KPI 4: Feedback Score — map Feedback text to numeric (Excellent=3, Average=2, Bad=1)
feedback_map = {'Excellent': 3, 'Average': 2, 'Bad': 1}
df['Feedback_Score'] = df['Feedback'].map(feedback_map)
print(df['Feedback_Score'].value_counts())

Feedback_Score
3.0    98174
2.0    60829
1.0    41940
Name: count, dtype: int64


In [44]:
# Verify all KPI columns
df[['Transaction_ID', 'Total_Amount', 'Total_Purchases', 'Revenue_per_Purchase',
    'High_Value_Flag', 'Ratings', 'Satisfied_Flag', 'Feedback', 'Feedback_Score']].head(10)

,Transaction_ID,Total_Amount,Total_Purchases,Revenue_per_Purchase,High_Value_Flag,Ratings,Satisfied_Flag,Feedback,Feedback_Score
0,8691788,324.09,3,108.03,0,5,1,Excellent,3.0
1,2174773,806.70,2,403.35,0,4,1,Excellent,3.0
2,6679610,1063.44,3,354.48,0,2,0,Average,2.0
3,7232460,2466.87,7,352.41,1,4,1,Excellent,3.0
4,4983775,248.56,2,124.28,0,1,0,Bad,1.0
5,6095326,1185.16,4,296.29,0,4,1,Good,NaN
6,5434096,630.12,2,315.06,0,1,0,Bad,1.0
7,2344675,46.59,1,46.59,0,1,0,Bad,1.0
8,4155845,2630.72,8,328.84,1,1,0,Bad,1.0
9,4926148,3976.10,10,397.61,1,4,1,Excellent,3.0


### Observation
Four row-level KPI columns have been added to the dataset.
`Revenue_per_Purchase` captures per-item basket value for each transaction.
`High_Value_Flag` marks the top 25% transactions by spend — useful for premium customer targeting in Tableau.
`Satisfied_Flag` enables a simple satisfaction rate KPI (% of orders rated 4 or 5).
`Feedback_Score` converts text feedback into a numeric field for aggregation and correlation in the dashboard.

## **Section 3 — Aggregated Summary Tables**

### **3a. Monthly Revenue Summary**

In [45]:
# Monthly revenue summary — total revenue, transaction count, avg amount by Month and Year
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

monthly_revenue = df.groupby(['Year', 'Month', 'Month_Num']).agg(
    Total_Revenue=('Total_Amount', 'sum'),
    Transaction_Count=('Transaction_ID', 'count'),
    Avg_Transaction_Value=('Total_Amount', 'mean'),
    Avg_Rating=('Ratings', 'mean'),
    Satisfaction_Rate=('Satisfied_Flag', 'mean')
).reset_index()

monthly_revenue['Avg_Transaction_Value'] = monthly_revenue['Avg_Transaction_Value'].round(2)
monthly_revenue['Avg_Rating'] = monthly_revenue['Avg_Rating'].round(2)
monthly_revenue['Satisfaction_Rate'] = (monthly_revenue['Satisfaction_Rate'] * 100).round(1)
monthly_revenue = monthly_revenue.sort_values(['Year', 'Month_Num']).reset_index(drop=True)

monthly_revenue

,Year,Month,Month_Num,Total_Revenue,Transaction_Count,Avg_Transaction_Value,Avg_Rating,Satisfaction_Rate
0,2023,March,3,34318881.42,24923,1377.00,3.17,49.0
1,2023,April,4,33199064.94,24073,1379.10,3.16,48.9
2,2023,May,5,33606955.52,24541,1369.42,3.16,49.1
3,2023,June,6,32877423.28,24135,1362.23,3.17,49.5
4,2023,July,7,33977289.80,24846,1367.52,3.16,49.1
5,2023,August,8,34238552.05,24802,1380.48,3.17,49.3
6,2023,September,9,33072415.60,24209,1366.12,3.17,49.4
7,2023,October,10,33744150.72,24931,1353.50,3.18,49.6
8,2023,November,11,32881474.57,23957,1372.52,3.16,48.8
9,2023,December,12,33822905.80,24670,1371.01,3.15,48.5


### **3b. Country / Region Summary**

In [46]:
# Country-level metrics — revenue, transaction count, avg rating, satisfaction rate
country_summary = df.groupby('Country').agg(
    Total_Revenue=('Total_Amount', 'sum'),
    Transaction_Count=('Transaction_ID', 'count'),
    Avg_Transaction_Value=('Total_Amount', 'mean'),
    Avg_Rating=('Ratings', 'mean'),
    Satisfaction_Rate=('Satisfied_Flag', 'mean'),
    High_Value_Transactions=('High_Value_Flag', 'sum')
).reset_index()

country_summary['Revenue_Share_Pct'] = (country_summary['Total_Revenue'] / country_summary['Total_Revenue'].sum() * 100).round(1)
country_summary['Avg_Transaction_Value'] = country_summary['Avg_Transaction_Value'].round(2)
country_summary['Avg_Rating'] = country_summary['Avg_Rating'].round(2)
country_summary['Satisfaction_Rate'] = (country_summary['Satisfaction_Rate'] * 100).round(1)
country_summary = country_summary.sort_values('Total_Revenue', ascending=False).reset_index(drop=True)

country_summary

,Country,Total_Revenue,Transaction_Count,Avg_Transaction_Value,Avg_Rating,Satisfaction_Rate,High_Value_Transactions,Revenue_Share_Pct
0,USA,1.269185e+08,93082,1363.51,3.05,44.9,23114,31.6
1,UK,8.458507e+07,61496,1375.46,3.13,48.7,15534,21.1
2,Germany,7.019501e+07,51293,1368.51,3.20,50.9,12852,17.5
3,Australia,6.012590e+07,43822,1372.05,3.28,52.9,10993,15.0
4,Canada,5.979528e+07,43775,1365.97,3.28,52.9,10873,14.9


### **3c. Product Category Summary**

In [47]:
# Product category metrics — revenue, transaction count, avg rating by category
category_summary = df.groupby('Product_Category').agg(
    Total_Revenue=('Total_Amount', 'sum'),
    Transaction_Count=('Transaction_ID', 'count'),
    Avg_Transaction_Value=('Total_Amount', 'mean'),
    Avg_Rating=('Ratings', 'mean'),
    Satisfaction_Rate=('Satisfied_Flag', 'mean'),
    Avg_Feedback_Score=('Feedback_Score', 'mean')
).reset_index()

category_summary['Revenue_Share_Pct'] = (category_summary['Total_Revenue'] / category_summary['Total_Revenue'].sum() * 100).round(1)
category_summary['Avg_Transaction_Value'] = category_summary['Avg_Transaction_Value'].round(2)
category_summary['Avg_Rating'] = category_summary['Avg_Rating'].round(2)
category_summary['Satisfaction_Rate'] = (category_summary['Satisfaction_Rate'] * 100).round(1)
category_summary['Avg_Feedback_Score'] = category_summary['Avg_Feedback_Score'].round(2)
category_summary = category_summary.sort_values('Total_Revenue', ascending=False).reset_index(drop=True)

category_summary

,Product_Category,Total_Revenue,Transaction_Count,Avg_Transaction_Value,Avg_Rating,Satisfaction_Rate,Avg_Feedback_Score,Revenue_Share_Pct
0,Electronics,95194317.69,69443,1370.83,3.27,52.0,2.34,23.7
1,Grocery,88784651.96,64984,1366.25,3.18,48.5,2.26,22.1
2,Clothing,72862811.71,53212,1369.29,3.11,48.2,2.26,18.1
3,Books,72546843.47,53047,1367.60,3.11,48.1,2.27,18.1
4,Home Decor,72231159.30,52782,1368.48,3.11,48.1,2.26,18.0


### **3d. Customer Segment Summary (Retention Table)**

In [48]:
# Customer segment metrics — revenue, avg spend, transaction count by segment
segment_summary = df.groupby('Customer_Segment').agg(
    Total_Revenue=('Total_Amount', 'sum'),
    Transaction_Count=('Transaction_ID', 'count'),
    Avg_Transaction_Value=('Total_Amount', 'mean'),
    Avg_Purchases_per_Txn=('Total_Purchases', 'mean'),
    Avg_Rating=('Ratings', 'mean'),
    Satisfaction_Rate=('Satisfied_Flag', 'mean'),
    High_Value_Transactions=('High_Value_Flag', 'sum')
).reset_index()

segment_summary['Revenue_Share_Pct'] = (segment_summary['Total_Revenue'] / segment_summary['Total_Revenue'].sum() * 100).round(1)
segment_summary['Avg_Transaction_Value'] = segment_summary['Avg_Transaction_Value'].round(2)
segment_summary['Avg_Purchases_per_Txn'] = segment_summary['Avg_Purchases_per_Txn'].round(2)
segment_summary['Avg_Rating'] = segment_summary['Avg_Rating'].round(2)
segment_summary['Satisfaction_Rate'] = (segment_summary['Satisfaction_Rate'] * 100).round(1)
segment_summary = segment_summary.sort_values('Total_Revenue', ascending=False).reset_index(drop=True)

segment_summary

,Customer_Segment,Total_Revenue,Transaction_Count,Avg_Transaction_Value,Avg_Purchases_per_Txn,Avg_Rating,Satisfaction_Rate,High_Value_Transactions,Revenue_Share_Pct
0,Regular,1.958709e+08,142951,1370.20,5.36,3.04,45.2,35793,48.8
1,New,1.211568e+08,88476,1369.37,5.38,3.20,50.5,22193,30.2
2,Premium,8.459205e+07,62041,1363.49,5.35,3.40,56.2,15380,21.1


### Observation
Four aggregated summary tables have been created for Tableau use.
The monthly summary includes `Month_Num` so Tableau can sort months in chronological order.
Country and category summaries include `Revenue_Share_Pct` to enable proportional comparisons directly in the dashboard without computed fields.
The segment summary acts as a retention reference table — showing how New, Regular, and Premium customers differ in transaction frequency and satisfaction.

## **Section 4 — Final Validation**

In [49]:
# Shape of final tableau-ready dataframe
print('Shape:', df.shape)

Shape: (293468, 33)


In [50]:
# Null check — should be zero nulls in all columns
null_check = df.isnull().sum()
print(null_check[null_check > 0] if null_check.sum() > 0 else 'No null values found in any column.')

Feedback_Score    92525
dtype: int64


In [51]:
# Dtypes check
df.dtypes

Transaction_ID                   int64
Customer_ID                      int64
City                            object
State                           object
Country                         object
Age                              int64
Gender                          object
Income                          object
Customer_Segment                object
Date                    datetime64[ns]
Year                             int64
Month                           object
Time                            object
Total_Purchases                  int64
Amount                         float64
Total_Amount                   float64
Product_Category                object
Product_Brand                   object
Product_Type                    object
Feedback                        object
Shipping_Method                 object
Payment_Method                  object
Order_Status                    object
Ratings                          int64
products                        object
Month_Num                

In [52]:
# Summary of all new columns added in this notebook
new_cols = ['Month_Num', 'Quarter', 'Day_of_Week', 'Hour',
            'Revenue_per_Purchase', 'High_Value_Flag', 'Satisfied_Flag', 'Feedback_Score']

print('New columns added in 05_final_load_prep:')
print('-' * 45)
for col in new_cols:
    print(f'  {col}: {df[col].dtype}  |  Sample: {df[col].iloc[0]}')

New columns added in 05_final_load_prep:
---------------------------------------------
  Month_Num: int32  |  Sample: 9
  Quarter: object  |  Sample: Q3
  Day_of_Week: object  |  Sample: Monday
  Hour: int32  |  Sample: 22
  Revenue_per_Purchase: float64  |  Sample: 108.03
  High_Value_Flag: int64  |  Sample: 0
  Satisfied_Flag: int64  |  Sample: 1
  Feedback_Score: float64  |  Sample: 3.0


In [53]:
# Final preview of tableau-ready dataframe
df.head()

,Transaction_ID,Customer_ID,City,State,Country,Age,Gender,Income,Customer_Segment,Date,...,Ratings,products,Month_Num,Quarter,Day_of_Week,Hour,Revenue_per_Purchase,High_Value_Flag,Satisfied_Flag,Feedback_Score
0,8691788,37249,Dortmund,Berlin,Germany,21,Male,Low,Regular,2023-09-18,...,5,Cycling shorts,9,Q3,Monday,22,108.03,0,1,3.0
1,2174773,69749,Nottingham,England,UK,19,Female,Low,Premium,2023-12-31,...,4,Lenovo Tab,12,Q4,Sunday,8,403.35,0,1,3.0
2,6679610,30192,Geelong,New South Wales,Australia,48,Male,Low,Regular,2023-04-26,...,2,Sports equipment,4,Q2,Wednesday,4,354.48,0,0,2.0
3,7232460,62101,Edmonton,Ontario,Canada,56,Male,High,Premium,2023-05-08,...,4,Utility knife,5,Q2,Monday,14,352.41,1,1,3.0
4,4983775,27901,Bristol,England,UK,22,Male,Low,Premium,2024-01-10,...,1,Chocolate cookies,1,Q1,Wednesday,16,124.28,0,0,1.0


### Observation
The final tableau-ready dataset has 0 null values and 33 columns (25 original + 8 newly derived).
All KPI flags and date features are correctly typed and populated.
The dataset is clean and ready for Tableau ingestion.

## **Section 5 — Export**

In [54]:
# Export main Tableau-ready dataset
TABLEAU_READY_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'
TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(TABLEAU_READY_PATH, index=False)
print(f'Saved: {TABLEAU_READY_PATH}')

Saved: /Users/utkarshjain/Desktop/SectionD_Team10_RetailAnalysis/data/processed/tableau_ready_dataset.csv
